# Introduction to AMSATs

In this notebook, you will connect to your AmSat via the [Secure Shell Protocol](https://en.wikipedia.org/wiki/Secure_Shell) (ssh for short). This will allow you to communicate remotely with the AmSat, and access telemetry data from the following sensors installed on the device:

* the **BME280**, which measures temperature, humidity, and pressure;
* the **INA219**, a power monitor which records voltage and current; and
* the **MPU6050**, a combined accelerometer and gyroscope.

## Pre-lab

Read the "Connecting to AMSATs" and "Reading Sensors" sections in your PHYC40870 Detector Lab manual. You should be familiar with ssh connections and command-line communications, as well as Python data structures and syntax.

# Basic Communication
## Exercise 1

Open a command-line terminal and connect to your AMSAT over ssh as discussed in the manual, and navigate to the `CubeSatSim` directory. There are two files we are interested in: `telem`, and `cubesatsim`. In the terminal, run the `telem` binary and copy and paste the output into the cell below, and provide a brief comment on the output.

### Answer: 

[Your `telem` output goes here]

Now run the `cubesatsim` binary. You will notice that, unlike `telem`, the output does not stop: it will keep going until you command the program to quit. Let the program run for a minute or so; to exit the program, press `ctrl+c` in the terminal to signal a keyboard interrupt. Examine the output (you may need to scroll up depending on how long it ran), and in the cell below identify and briefly describe three or four components of the data outputs. Don't worry if you don't understand what this output means just yet; we are only concerned here with recognizing important features in the data.

### Answer:

[Your response goes here]

## Exercise 2
Now you will use the code snippets in the `Capturing Output` section of the manual as templates to accomplish some tasks and answer a few questions. 

#### 1: Redirect both output streams from `telem` to appropriately named files (e.g. `telem_stdout.txt` and `telem_stderr.txt`. Comment in the cell below on how `telem` communicates its information. Use the `sftp` command to copy these files to your working directory on the lab computer.

### Answer:

[Your response here]

#### 2: Repeat Part 1 for the `cubesatsim` binary, and comment in the cell below on how the binary communicates its information.

### Answer: 

[Your response here]

# Interfacing using Python

We can interact with the AMSAT using Python as well, as the `paramiko` package is specially designed for SSH communication. We could feasibly do just about everything we need through the terminal interface above, but using Python allows us to handle the communication, the data parsing, and the data analysis all in one place.

First we need to import a few packages.

In [1]:
from datetime import datetime
import csv
import os, re
import time
import paramiko

We also might want to have a logger, which can print useful information to the terminal as well as write "debugging" information to the log file. We have provided a logging utility for you that can be called simply by importing the `event_logger` package.

In [2]:
from event_logger import logging

2026-11-24 13:51:17,350 - root - INFO - Logging configuration initialized.


Now we need to set up a basic configuration so our local machine can figure out how to establish and authenticate the "session" or connection with the AMSAT. It is important to note that it is **strongly discouraged** to store sensitive authentication information in hard-coded, plain text variables like this; we are doing it here only because there is nothing sensitive about our setup. Ideally, we would want to store our credentials in a secret configuration file.

In [5]:
router_ip = "10.199.170.48"   # replace this with your amsat number, ie cubesatsim1.local
router_username = "pi"
router_password = "raspberry"
port_number = 22
command_path = "/home/pi/CubeSatSim/telem"
csv_file = "./telem_output.csv"

In [6]:
# We configured our logger, now let's import it so we can choose what we print
logger = logging.getLogger()    

## Establishing a connection 
The basic syntax in `paramiko` for establishing an ssh connection is discussed in their online documentation. We want to first create an ssh "client" or representation of an actually ssh session on a remote server. Then we load the ssh keys and tell the client to establish the connection.

```python
ssh = paramiko.SSHClient() 
ssh.load_system_host_keys()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect([arguments here])
```
We will put this all together into a function that returns the connected ssh instance. When we want to close the connection, we simply type

```python
ssh.close()
```

In [7]:
def ssh_connect():
    """
    Create an 'instance' or ssh object that connects us to the AmSat 
    """

    _ssh = paramiko.SSHClient()
    _ssh.load_system_host_keys()
    _ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    try:
        _ssh.connect(
            router_ip,
            port=port_number,
            username=router_username,
            password=router_password,
            look_for_keys=False
        )
        logger.info(f"Connected to {router_ip}")
        return _ssh
    except Exception as e:
        logger.error(f"Error occured: {e}")
        return None
        

## Executing commands

We can send commands to the remote device using the ```ssh.exec_command()``` method, which returns the `stdin`, `stdout`, and `stderr` streams.

```python
stdin, stdout, sterr = ssh.exec_command('ls', get_pty=False)
```
The `get_pty=False` tells the client that we don't want an interactive terminal; just execute the script and finish. These streams need to be read and decoded before we can do anything useful with them.
```python
output = stdout.read().decode('utf-8')
```
As before, we want this to be modular so we'll put it into a function. 

In [8]:
def run_remote_command(ssh_obj, command_path):
    """
    Using an ssh instance, execute a command and get the output
    """

    date_command = "+\"%Y-%m-%dT%H:%M:%S.%3N%:z\""
    remote_command = f'bash -c -l "date {date_command} && {command_path} && date {date_command} 2>&1"'

    ssh_stdin, ssh_stdout, ssh_stderr = ssh_obj.exec_command(
        remote_command,
        get_pty=False
    )
    raw_output = ssh_stdout.read().decode('utf-8').strip()
    logger.info(f"Output collected from remote command: {command_path}")
    errors = ssh_stderr.read().decode('utf-8').strip()
    if errors:
        logger.error(f"Error encountered: {errors}")

    return raw_output

A few things to notice here. First, we do not use a slash-dot to tell the AMSAT what to do; instead we use `bash` and provide the `-c` (execute a command from a string) and `-l` (operate as a login shell) arguments before our `remote_command`. (The reason for this is esoteric; just trust us that it works.) 

You will also notice that the `remote_command` variable is a bit more complex than what we might have been expecting. This is because our `telem` binary doesn't return any time information, so we get the time before and after the binary is executed. 

## Clock offset and delay

The timestamps in our `telem` data come from the AMSAT's own clock (`date` runs on the Pi), and that clock is not necessarily in sync with the lab computer. `delay_sample()` asks the AMSAT for its time once and records the round-trip time (the delay) and the clock offset; `measure_clock_offset()` repeats it and keeps the most precise sample. `time_remote_command()` uses bash's `time` keyword to separate how long a command actually runs on the AMSAT from the delay added by ssh, the network, and starting the login shell.

`run()` and `run_cubesatsim_total()` use these to log the delay while measuring (to a `..._delay.csv` file next to the data), so we can see whether the delay grows when the AMSAT is busy.

In [27]:
import threading


def delay_sample(ssh_obj):
    """
    One round trip to the AMSAT: ask for its clock with `date` and time how long
    the answer takes to come back.

      rtt_s    - round-trip time, i.e. the delay (ssh + network + the AMSAT
                 starting a process, which is slower when its CPU is busy)
      offset_s - AMSAT clock minus computer clock, good to +/- rtt_s / 2

    Returns a dict, or None if the reply can't be read.
    """
    t_send = time.time()
    _, ssh_stdout, _ = ssh_obj.exec_command("date +%s.%N", get_pty=False)
    reply = ssh_stdout.read().decode('utf-8').strip()
    t_recv = time.time()

    try:
        # seconds since 1970, the same scale as time.time()
        t_amsat = float(reply.splitlines()[-1])
    except (ValueError, IndexError):
        logger.warning(f"Could not read the AMSAT time from: {reply!r}")
        return None

    # assume the AMSAT read its clock halfway through the round trip
    t_mid = (t_send + t_recv) / 2
    return {
        "computer_time": datetime.fromtimestamp(t_mid).isoformat(timespec="milliseconds"),
        "amsat_time": datetime.fromtimestamp(t_amsat).isoformat(timespec="milliseconds"),
        "rtt_s": round(t_recv - t_send, 4),
        "offset_s": round(t_amsat - t_mid, 4),
    }


def measure_clock_offset(ssh_obj, n_samples=10):
    """
    Estimate how far the AMSAT's clock is from this computer's clock.

    Each delay_sample() is only good to +/- half its round-trip time, so we take
    several and keep the one with the shortest round trip.

    A positive offset means the AMSAT's clock is AHEAD of the computer's.
    """
    samples = [s for s in (delay_sample(ssh_obj) for _ in range(n_samples)) if s]
    if not samples:
        logger.error("No valid time samples from the AMSAT.")
        return None

    best = min(samples, key=lambda s: s["rtt_s"])
    best["uncertainty_s"] = round(best["rtt_s"] / 2, 4)

    logger.info(f"AMSAT time:    {best['amsat_time']}")
    logger.info(f"Computer time: {best['computer_time']}")
    logger.info(f"Clock offset (AMSAT - computer): {best['offset_s']:+.3f} s "
                f"+/- {best['uncertainty_s']:.3f} s (best of {len(samples)} samples)")
    return best


def time_remote_command(ssh_obj, command):
    """
    Split the time it takes to run `command` on the AMSAT into:
      exec_s     - how long the command itself runs on the AMSAT (bash `time`)
      total_s    - how long this computer waits for it
      overhead_s - total_s - exec_s: the delay from ssh, the network and
                   starting the login shell
    """
    # bash's `time` prints the elapsed (real) seconds to stderr; the command's
    # own output goes to /dev/null so only that number comes back
    remote_command = f"bash -c -l 'TIMEFORMAT=%R; time {command} > /dev/null 2>&1'"

    t_send = time.time()
    _, ssh_stdout, ssh_stderr = ssh_obj.exec_command(remote_command, get_pty=False)
    ssh_stdout.read()   # wait until the command has finished
    timing = ssh_stderr.read().decode('utf-8').strip()
    t_recv = time.time()

    try:
        exec_s = float(timing.splitlines()[-1].replace(',', '.'))
    except (ValueError, IndexError):
        logger.error(f"Could not read the timing output: {timing!r}")
        return None

    total_s = t_recv - t_send
    result = {"exec_s": exec_s, "total_s": total_s, "overhead_s": total_s - exec_s}

    logger.info(f"{command}: runs for {exec_s:.3f} s on the AMSAT, {total_s:.3f} s "
                f"round trip -> {result['overhead_s']:.3f} s delay")
    return result


def delay_monitor(stop, busy, rows, interval=2.0):
    """
    Meant to run in a background thread: every `interval` seconds take one
    delay_sample() on its own ssh connection and append it to `rows`, until
    `stop` is set. The 'cubesatsim_running' column records whether `busy` was
    set when the sample was taken.
    """
    ssh_d = ssh_connect()
    if ssh_d is None:
        return
    try:
        while not stop.is_set():
            running = busy.is_set()
            d = delay_sample(ssh_d)
            if d:
                d["cubesatsim_running"] = running
                rows.append(d)
            stop.wait(interval)   # like time.sleep(), but wakes up as soon as stop is set
    finally:
        ssh_d.close()


def summarize_delay(rows, label=""):
    """Log the mean, min and max delay of a list of delay samples."""
    rtts = [r["rtt_s"] for r in rows]
    if rtts:
        logger.info(f"{label}delay mean {sum(rtts) / len(rtts):.3f} s, min {min(rtts):.3f} s, "
                    f"max {max(rtts):.3f} s ({len(rtts)} samples)")


def write_delay_csv(filename, rows):
    """Write a list of delay samples to {filename}.csv"""
    if not rows:
        logger.warning("No delay samples to write.")
        return
    with open(f"{filename}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader()
        w.writerows(rows)
    logger.info(f"Wrote {len(rows)} delay samples to {filename}.csv")

In [33]:
ssh_t = ssh_connect()

clock = measure_clock_offset(ssh_t, n_samples=100)
telem_timing = time_remote_command(ssh_t, command_path)

ssh_t.close()

2026-11-24 14:07:55,536 - paramiko.transport - INFO - Connected (version 2.0, client OpenSSH_8.4p1)
2026-11-24 14:08:00,458 - paramiko.transport - INFO - Authentication (password) successful!
2026-11-24 14:08:00,461 - root - INFO - Connected to 10.199.170.48
2026-11-24 14:08:13,765 - root - INFO - AMSAT time:    2026-11-24T14:08:11.213
2026-11-24 14:08:13,768 - root - INFO - Computer time: 2026-11-24T14:08:11.060
2026-11-24 14:08:13,769 - root - INFO - Clock offset (AMSAT - computer): +0.153 s +/- 0.032 s (best of 100 samples)
2026-11-24 14:08:16,631 - root - INFO - /home/pi/CubeSatSim/telem: runs for 2.564 s on the AMSAT, 2.860 s round trip -> 0.296 s delay


## Exercise 3
In the cell below, create the ssh instance, run the `telem` binary, and print the output. (Hint: pay close attention to the output structure; you might want to use a `for`-loop to print it.)

In [33]:
ssh_1 = ssh_connect()


2026-11-23 14:45:03,230 - paramiko.transport - INFO - Connected (version 2.0, client OpenSSH_8.4p1)
2026-11-23 14:45:08,141 - paramiko.transport - INFO - Authentication (password) successful!
2026-11-23 14:45:08,143 - root - INFO - Connected to 10.199.170.48


In [8]:
telem_output = run_remote_command(ssh_1, command_path)
# cubesatsim_output = run_remote_command(ssh_1, "/home/pi/CubeSatSim/cubesatsim")

2026-11-17 11:35:54,893 - root - INFO - Output collected from remote command: /home/pi/CubeSatSim/telem


In [9]:
print(cubesatsim_output)

NameError: name 'cubesatsim_output' is not defined

In [7]:
# print(type(telem_output))
table = []


lines  = telem_output.split('\n')
for i, line in enumerate(lines):
    if i >= 6 and i <= 13:
        table.append(line)
        
        
        
table2=[]  
#header: dict[str, list[str]] = {} 
for line in table:
    #stripped = line.split("|")
   # stripped[1] = stripped[1].split("V")

    stripped = line.replace(" ", "").replace("mA","").split("|")
#     stripped = stripped.split("|")
#     stripped[1] = stripped[1].split("V")
    row = [stripped[0]] + stripped[1].split("V") 
#    row = row +
    
    
    table2.append(row)
    
 



print(table2)
    
# with open(csv_file, 'w') as csvfile:
#     csv_file1 = csv.writer(csvfile,delimiter=' ')
#     csv_file1.writerows(table2)

# with open(csv_file, 'r') as csvfile:
#     csv_file1 = csv.reader(csvfile,delimiter=',')
#     for line in csv_file1:
#         print(line)

NameError: name 'telem_output' is not defined

In [ ]:
print(telem_output)


### Cleaning and parsing output

We have redirected the standard streams to Python variables; now we should process the output and save it as structured data. We need to pay attention to the format of our output. We have some strings ("SSH is enabled..." and "This is a security risk..." for example) that we can skip, we have two timestamps (before and after the binary is run), and we have data with units and parameter names. 

One approach to this is to adjust our `run_remote_command()` function to also "sanitize" our output (that is, prepare it for parsing) before returning it. This is, arguably, not very Pythonic though, since we would be assigning multiple duties to a single function. A cleaner approach would be to write a few smaller functions that will sanitize the output, and then parse it properly.

## Exercise 4
In the cell below, complete the `sanitize_stdout_stream()` function. You should identify patterns in the `telem` output that you wish to skip, check each output line to see if those patterns are in the line, and append only the "good" lines to the `sanitized_output` list.

(Hint: use `break` and `continue` to control the parsing.)

In [20]:
def sanitize_stdout_stream(raw_stdout):
    """
    Take a full, unprocessed stdout stream and sanitize it.
    """

    patterns_to_skip = ["SSH is enabled and the default password for the 'pi' user has not been changed.",
                        "This is a security risk - please login as the 'pi' user and type 'passwd' to set a new password.",
                         'CubeSatSim v1.2 INA219 Voltage and Current Telemetry',  ''

        # patterns in the output we know we don't want
        # e.g. ["pattern one", "pattern two", ...]
    ]

    sanitized_output = []

    # finish this code
    for line in raw_stdout.splitlines():            
        skip_line = False
        if line in patterns_to_skip:
            skip_line = True
        else:
            sanitized_output.append(line)
        if skip_line:
            continue

            
            
            
    logger.info(f"Sanitized {len(sanitized_output)} data lines.")

    return sanitized_output

In [ ]:
a = sanitize_stdout_stream(telem_output)
print(a)

In [22]:
def parse_data(a):
    
    data={}
    data['time_start'] = a[0]
    data['time_end'] = a[-1]

    for row in a[1:-1]:
        header, values = row.split("|")
        voltage,_, current,_ = values.split()

    #     data[header.strip()] = {"V": float(voltage), "mA":float(current)}
        data[f"{header} (V)"] = float(voltage)
        data[f"{header} (mA)"] = float(current)
    # print(data) 
    return data
    

In [ ]:
a = parse_data(a)
print(a)

This gives us a cleaner output that will be easier to parse. The parsing for this particular output is a bit complex because there are two timestamps for each sensor readout. We use these to compute the loop time, but it means that we need to loop over the output twice: once to find the timestamps and compute the loop time, and once to find the data lines and put it all into a list. To save you the time, we have written part of the the parser for you. 

## Exercise 5

In the code below, finish the `parse_sanitized_output()` function, beginning with 

```python
for i, line in enumerate(raw_telemetry_lines):

```

In [10]:
def parse_sanitized_output(sanitized_output):
    """
    Takes a sanitized output and produces a list of structured data.
    """

    timestamps = []
    raw_telemetry_lines = []

    # check if it's a timestamp
    for line in sanitized_output:
        try:
            datetime.fromisoformat(line)
            timestamps.append(line)
        except ValueError:
            raw_telemetry_lines.append(line)

    if len(timestamps) == 2:
        try:
            start_ts, end_ts = timestamps
            dt_start = datetime.fromisoformat(start_ts)
            dt_end = datetime.fromisoformat(end_ts)
            
            loop_time = (dt_end - dt_start).total_seconds()
            
            logger.debug(f"Extracted timestamps: {start_ts}, {end_ts}")
            logger.debug(f"Calculated loop time: {loop_time:0.3f} seconds")
        except Exception as e:
            logger.warning(f"Unable to parse line: {line}")
            pass
    else:
        logger.warning(f"Not enough timestamps.")
        loop_time, start_ts = None, None

    # now parse the data
    parsed_rows = []
    
    for i, line in enumerate(raw_telemetry_lines):
        # your code here
        
        logger.info(f"Successfully parsed {len(parsed_rows)} telemetry data lines.")
    return parsed_rows


In [ ]:
parse_sanitized_output(a)

## Exercise 6
Finally we can write the output to a file. Complete the `update_csv()` function below. The function should take your parsed data, a filename, and header row as arguments, and check if the file already exists using `os.path.exists()`. If the does not exist, then you should first write the header row before writing the data; if the file does exist, then simply write the data. (Remember you can avoid overwriting an existing file using `open(file, 'a')`.) 

In [24]:
def write_csv(filename, data):
#     print(data)
    with open(f"{filename}.csv", "w") as f:
        w = csv.DictWriter(f, fieldnames = data.keys())
        w.writeheader()
        w.writerow(data)
    return None

In [37]:
write_csv('telem_output1', a)

NameError: name 'a' is not defined

In [25]:
def update_csv(filename,data):
    """ If there is a csv file, append to it; otherwise write to a new one """
    # complete the code here
    with open(f"{filename}.csv", "a") as f:
        w = csv.DictWriter(f, fieldnames = data.keys())
        w.writerow(data)
    
    

In [ ]:
update_csv(a, 'telem_output1')

A nice feature in Jupyter notebooks is that we can actually use shell commands from a cell. This is done by typing the command we'd normally use in a command-line terminal, but prefaced with an exclamation point, e.g. `!pwd` will print our current working directory. Since we have written some data to the `telem.csv` file, we might want to check that it all looks how we'd expect. We can always open the folder and click on the file to open it in a text editor -- but we can also use the `cat` command to print the output of that file to our screen!

In [ ]:
!cat telem_output1.csv

If our file is really long we might want to use `head telem.csv` or `tail telem.csv` to only print the first few lines or the last few lines. Let's close our ssh connection now and move on.

In [15]:
ssh_1.close()

# Task 1

Now you have all the pieces. In the cell below, write your own code that calls the `telem` binary in a loop, cleans and parses the output, and writes the output from each loop to a CSV file. Your code should use `try-except-finally` logic to run a `while`-loop which exits on a `KeyboardInterrupt` exception and, if everything goes wrong, finally closes the `ssh` connection. Your `while`-loop should include a `time.sleep()` statement, and you should also keep track of the number of iterations. 

**Hints:** you only need to establish the `ssh` connection _once_, so it does not need to be included in the loop; the connection can be closed with `ssh.close()`.

In [10]:
# your code here
def run():
    command_path = "/home/pi/CubeSatSim/telem"
    ssh = ssh_connect()
    i =0
    try:
        while True:
            telem_output = run_remote_command(ssh, command_path)
            san_table = sanitize_stdout_stream(telem_output)
            parsed_table = parse_data(san_table)
            if i== 0:
                write_csv('telem_output',parsed_table)
            else:
                update_csv('telem_output',parsed_table)
            time.sleep(3)
            i+=1
    except KeyboardInterrupt:
        ssh.close()
    finally:
        print("Interrupted")
        print(f"{i} iterations")

In [18]:
def run(filename, track_delay=True):
    command_path = "/home/pi/CubeSatSim/telem"
    ssh = ssh_connect()

    i = 0
    start_time = time.time()
    delay_rows = []

    try:
        while time.time() - start_time < 60:
            telem_output = run_remote_command(ssh, command_path)
            san_table = sanitize_stdout_stream(telem_output)
            parsed_table = parse_data(san_table)

            if i == 0:
                write_csv(filename, parsed_table)
            else:
                update_csv(filename, parsed_table)

            # one delay measurement per loop, saved to {filename}_delay.csv at the end
            if track_delay:
                d = delay_sample(ssh)
                if d:
                    d["iteration"] = i
                    delay_rows.append(d)

            time.sleep(3)
            i += 1

    except KeyboardInterrupt:
        pass

    finally:
        ssh.close()
        if track_delay:
            write_delay_csv(f"{filename}_delay", delay_rows)
            summarize_delay(delay_rows)
        print("Interrupted")
        print(f"{i} iterations")
        print(f"Runtime: {time.time() - start_time:.1f} s")

In [ ]:
run('Measurements/telem_control')

# Task 2

Capturing the output in Python for the `cubesatsim` binary is more complicated than the `telem` binary. Because `cubesatsim` returns a continuous stream of data, the `stdout` and `stderr` streams are not accessible until an interrupt is sent to the AMSAT. A workaround is to use `get_pty=True` as a parameter to our `ssh.exec_command()` function, which creates an "interactive session"  with the AMSAT. In a standard terminal, we would type `ctrl+c` to end the program, which we can do here by sending `stdin.write(chr(3))` or `stdin.write(b'\x03')` to the AMSAT after some period of time. Because this is a virtual terminal, though, both the standard output and standard error streams are merged into a single stream, which is send to our `stdout` variable in our function.

To help you get started, we have provided a function for you called `run_cubesatsim()`, which returns the full output stream as a variable called `ssh_chunks`.

In the cells below, write a function or functions to parse the output and extract the BME280 and MPU6050 sensor data, which should be written to their own CSV files. (If necessary, adapt your `update_csv()` function to account for this, rather than writing two separate functions.) You should set a longer read duration (e.g. 600 s) to ensure you gather enough data.

### Hints
You can be clever with how you extract the data. Recall from above that in each loop of the `cubesatsim` binary, there can be multiple sensor readings; but only one sensor reading is consolidated into a telemetry string, which means you can simplify your parsing significantly.

The BME280 output is printed in the order:

`Temperature Pressure Altitude Humidity`
  
and the MPU6050 is printed in the order: 

`Gyro_X Gyro_Y Gyro_Z Accel_X Accel_Y Accel_Z`

Pay close attention to the _structure_ of the output. Because `run_cubesatsim()` reads data in _chunks_, some items in the `ssh_chunks` variable are actually multiple logical lines combined. If you are feeling adventurous, you can use the Python `re` module, but a less esoteric approach could do the following:

* Remove the `\r` and `\n` characters in each line with `line.replace('\r', '').replace('\n', '')`
* Make sure only a single whitespace is present between words with `' '.join(line.split())`
* Split the cleaned lines into parts with `line.split(' ')` if a specific pattern is present in the cleaned lines

You would of course do this in a loop over all the lines in the output. In this way, it will take a line of string, look for a pattern, and convert the string to a list if that pattern is there, e.g. `"This is a string"` becomes `["This", "is", "a", "string"]`. You can then search for specific patterns in each list, and also keep track of the index of each element in the list and extract them as you need. 

We have provided some skeleton code to help you get started.

In [14]:
def run_cubesatsim(ssh, chunk_size=4096, read_duration=300):
    try:
        remote_command = f'bash -c -l "/home/pi/CubeSatSim/cubesatsim"'
        stdin, stdout, _ = ssh.exec_command(
            remote_command,
            get_pty=True
        )

        ssh_chunks = []
    
        start_time = time.time()
        logger.info(f"Read duration set to: {read_duration} seconds")
    
        loop_counter = 0
        while True:
            if stdout.channel.recv_ready():
                logger.debug("Stdout buffer ready to be read")
                data = stdout.channel.recv(chunk_size).decode('utf-8')
                if data:
                    ssh_chunks.append(data)
    
            if time.time() - start_time > read_duration:
                break
            time.sleep(0.1)
            loop_counter += 1

        # when the loop is done, send the ctrl+c command and close the shell
        stdin.write(chr(3))
        stdin.flush()
        time.sleep(1)
    
        next_time = time.time() - start_time
        logger.info(f"{loop_counter} loops done.")
        logger.info(f"{len(ssh_chunks)} output lines read in {next_time:0.3f} seconds. Cleaning up...")

        # read whatever is left in the buffer
        while stdout.channel.recv_ready():
            data = stdout.channel.recv(chunk_size).decode('utf-8')
            if data:
                ssh_chunks.append(data)
    
        exit_status = stdout.channel.recv_exit_status()
        end_time = time.time() - start_time
        logger.info(f"Exit status received: {exit_status}")
        logger.info(f"Ending loop after {end_time:0.3f} seconds.")
    except Exception as e:
        logger.warning(f'Some error occurred: {e}')

    return ssh_chunks

In [16]:
ssh = ssh_connect()

chunks_output = run_cubesatsim(ssh, chunk_size=4096, read_duration=300)

2026-11-23 14:33:31,678 - root - ERROR - Error occured: [Errno 11001] getaddrinfo failed
2026-11-23 14:33:31,680 - root - WARNING - Some error occurred: 'NoneType' object has no attribute 'exec_command'


UnboundLocalError: cannot access local variable 'ssh_chunks' where it is not associated with a value

In [ ]:
chunks_output

In [17]:
def parse_cubesatsim(raw_chunks):
    """
    skeleton code to help you get started
    NOTE: You must comment your code!
    
    Remove the \r and \n characters in each line with line.replace('\r', '').replace('\n', '')
    Make sure only a single whitespace is present between words with ' '.join(line.split())
    Split the cleaned lines into parts with line.split(' ') if a specific pattern is present in the cleaned lines

    """

    bme_data = []
    mpu_data = []
    chunk_data = []
    # you can use relative time rather than timestamps
    current_time = 0

    loop_pattern = "Loop time: "
    telemetry_pattern = "Python read Result:"
    
    for chunk in raw_chunks:
        # you may need to clean the stream
        chunk = chunk.replace('\r', '').replace('\n', '')
        chunk = " ".join(chunk.split())
        
        if loop_pattern in chunk:
            looptime = chunk.split()
            
            
            j = looptime.index("time:")
           
            
           
            time = float(looptime[j+1])
            #print(time)
            pass
        
        
        if telemetry_pattern in chunk:
            parts = chunk.split()
            if "BME280" in parts:
                i = parts.index("BME280")
                pattern = []
                pattern.append(time)
                print(time)
                pattern.append(float(parts[i+1]))
                pattern.append(float(parts[i+2]))
                pattern.append(float(parts[i+3]))
                pattern.append(float(parts[i+4]))
                bme_data.append(pattern)
                
            if "MPU6050" in parts:
                i = parts.index("MPU6050")
                pattern2 = []                
                pattern2.append(time)
                pattern2.append(float(parts[i+1]))
                pattern2.append(float(parts[i+2]))
                pattern2.append(float(parts[i+3]))
                pattern2.append(float(parts[i+4]))
                pattern2.append(float(parts[i+5]))
                pattern2.append(float(parts[i+6]))
                mpu_data.append(pattern2)
            
            pass
        
        
        chunk_data.append(chunk)
    return bme_data, mpu_data, chunk_data
    

In [18]:
bme_data,mpu_data,parsed_chunk = parse_cubesatsim(chunks_output)

NameError: name 'chunks_output' is not defined

In [ ]:
parsed_chunk

In [ ]:
mpu_data


In [ ]:
bme_data

In [15]:
def write_csv_cubesatsim(filename, data, headers):
    print(data)
    with open(f"{filename}.csv", "w") as f:
        
        w = csv.writer(f)
        w.writerow(headers)
        w.writerows(data)
            
    return None

In [ ]:
write_csv_cubesatsim("mpu_data",mpu_data,["time","x","y","z","x2","y2","z2"])

In [ ]:
write_csv_cubesatsim("mpu_data",mpu_data,["x","y","z","x2","y2","z2"])
write_csv_cubesatsim("bme_data",bme_data,["x","y","z","x2","y2","z2"])

update_csv(a, 'telem_output1')


In [16]:
def run_cubesatsim_total(bme_name, mpu_name, chunk_size=4096, read_duration=300,
                         track_delay=True, delay_interval=2.0, idle_time=20):
    ssh = ssh_connect()

    if track_delay:
        # Measure the delay in the background on a second ssh connection:
        # for idle_time seconds before cubesatsim starts, while it runs, and for
        # idle_time seconds after it stops, so idle and busy can be compared.
        stop, busy, delay_rows = threading.Event(), threading.Event(), []
        monitor = threading.Thread(target=delay_monitor,
                                   args=(stop, busy, delay_rows, delay_interval),
                                   daemon=True)
        monitor.start()
        time.sleep(idle_time)
        busy.set()

    chunks_output = run_cubesatsim(ssh, chunk_size=chunk_size, read_duration=read_duration)

    if track_delay:
        busy.clear()
        time.sleep(idle_time)
        stop.set()
        monitor.join()
        write_delay_csv(f"Measurements/{bme_name}_delay", delay_rows)
        summarize_delay([r for r in delay_rows if not r["cubesatsim_running"]], "cubesatsim idle:    ")
        summarize_delay([r for r in delay_rows if r["cubesatsim_running"]], "cubesatsim running: ")

    bme_data,mpu_data,parsed_chunk = parse_cubesatsim(chunks_output)
    # full_output = "".join(chunks_output)

    # print(full_output)
    # bme_data,mpu_data, _ = parse_cubesatsim(ssh_chunks)
    write_csv_cubesatsim(f"Measurements/{mpu_name}",mpu_data,["time","acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"])
    write_csv_cubesatsim(f"Measurements/{bme_name}",bme_data,["time","Temp","Press","Alt","Hum"])
    ssh.close()
    return None


In [17]:
run_cubesatsim_total('delay_test', 'delay_test',read_duration=60)

2026-11-24 13:53:11,761 - paramiko.transport - INFO - Connected (version 2.0, client OpenSSH_8.4p1)
2026-11-24 13:53:16,673 - paramiko.transport - INFO - Authentication (password) successful!
2026-11-24 13:53:16,675 - root - INFO - Connected to 10.199.170.48
2026-11-24 13:53:17,292 - paramiko.transport - INFO - Connected (version 2.0, client OpenSSH_8.4p1)
2026-11-24 13:53:22,317 - paramiko.transport - INFO - Authentication (password) successful!
2026-11-24 13:53:22,319 - root - INFO - Connected to 10.199.170.48
2026-11-24 13:53:36,723 - root - INFO - Read duration set to: 60 seconds
2026-11-24 13:54:37,806 - root - INFO - 597 loops done.
2026-11-24 13:54:37,808 - root - INFO - 40 output lines read in 61.083 seconds. Cleaning up...
2026-11-24 13:54:37,810 - root - INFO - Exit status received: -1
2026-11-24 13:54:37,812 - root - INFO - Ending loop after 61.087 seconds.
2026-11-24 13:54:57,818 - root - INFO - Wrote 45 delay samples to Measurements/delay_test_delay.csv
2026-11-24 13:54:57

NameError: name 'parse_cubesatsim' is not defined

In [26]:
run('Measurements/telem_delay_test')

2026-11-24 13:58:44,495 - paramiko.transport - INFO - Connected (version 2.0, client OpenSSH_8.4p1)
2026-11-24 13:58:49,373 - paramiko.transport - INFO - Authentication (password) successful!
2026-11-24 13:58:49,376 - root - INFO - Connected to 10.199.170.48
2026-11-24 13:58:54,298 - root - INFO - Output collected from remote command: /home/pi/CubeSatSim/telem
2026-11-24 13:58:54,300 - root - INFO - Sanitized 10 data lines.
2026-11-24 13:58:59,948 - root - INFO - Output collected from remote command: /home/pi/CubeSatSim/telem
2026-11-24 13:58:59,950 - root - INFO - Sanitized 10 data lines.
2026-11-24 13:59:05,506 - root - INFO - Output collected from remote command: /home/pi/CubeSatSim/telem
2026-11-24 13:59:05,509 - root - INFO - Sanitized 10 data lines.
2026-11-24 13:59:10,907 - root - INFO - Output collected from remote command: /home/pi/CubeSatSim/telem
2026-11-24 13:59:10,909 - root - INFO - Sanitized 10 data lines.
2026-11-24 13:59:16,655 - root - INFO - Output collected from rem

Interrupted
10 iterations
Runtime: 61.9 s
